# Phase 12 — Controlled Ablation Training

Manual execution only. The notebook is delivered with training disabled. Preparation must not execute any experimental epoch or optimizer update.

## Public-copy execution policy
This is an English rendering of a historical research notebook. Outputs and execution counts were cleared; no cell was rerun for Phase 18. Historical findings below describe earlier runs only. Every code cell is gated, including setup cells that may perform model checks, preprocessing or writes. Enable `CONFIRM_PUBLIC_WORKFLOW` deliberately in the first code cell only after reviewing [the reproduction guide](../docs/reproducibility.md). Existing stage-specific confirmations must also be enabled separately. Run from the repository root or set `MORPHONET_ROOT` to a private writable experiment workspace with the same source/config layout; set `MORPHONET_DATA_ROOT` for dataset discovery. Original source/configuration fingerprints, private split manifests and checkpoints are not portable artifacts; do not bypass their validation. See the guide for prerequisites and limitations.


## Objective and scientific protocol

Prepare A0–A4 on five frozen patient-level folds: 25 future runs. A5 is the existing frozen full MorphoNet-BT experiment and is rejected by the training launcher. No architectures, targets, splits or hyperparameters are changed. Single-fold validation does not establish component effectiveness; final conclusions require the separately approved Phase 13 held-out evaluation and OOF aggregation.

## Manual execution policy

Only the explicitly marked manual launch cell can initiate an experiment. It requires the user to manually set `CONFIRM_FULL_TRAINING = True`. The initial selection is A0 Fold 1, confirmation false, resume false and overwrite false. Preparation performs no epochs, backward passes or optimizer steps. Phase 11 memory diagnostics are reused.

## Imports, reproducibility and project paths

In [ ]:
CONFIRM_PUBLIC_WORKFLOW = False  # Explicit opt-in; no cell has been rerun.
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
import os, sys, importlib
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
ROOT = Path(__import__("os").environ.get("MORPHONET_ROOT", str(Path.cwd()))).resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0,str(ROOT))
import pandas as pd
from IPython.display import display
from src.ablation.validation import prepare
from src.ablation.protocol import check_protection
from src.ablation import training as ablation_training
pd.set_option('display.max_columns', 15)
pd.set_option('display.precision', 5)

## Frozen variant definitions

A0: conventional stem and original local-only processing. A1: add MSS. A2: add privileged morphology supervision. A3: add MRI-predicted boundary attention. A4: add contextual global branch with arithmetic-mean fusion. A5 is excluded from training and reuses its original five best checkpoints. Encoder widths, depths, classifier, preprocessing and architecture sources are protected by Phase 11 hashes.

## Frozen split verification and Test-lock audit

**Test data are locked during Phase 12 and will only be accessed during the dedicated Phase 13 evaluation.**

Only split CSV identity metadata is read to audit Train/Validation/Test PID separation and exact frozen hashes. No Test Dataset, Test DataLoader, Test image, Test prediction or performance parser is used. Existing evaluation files are hashed as opaque bytes solely for artifact protection.

## Protected artifacts and non-training preparation

Protect original main checkpoints/results, baseline checkpoints/results, evaluation artifacts, all splits and Phase 11 setup files. No ablation experimental directory, checkpoint, history or summary is created during preparation. Safe checks cover model construction, loss dispatch, invalid selections and in-memory state serialization. Experimental epoch and AdamW-step entry points are patched to fail if accidentally called.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
info, preparation_audit, readiness = prepare(ROOT)
display(readiness)
display(info['split_audit'])
print('Training configuration fingerprint:', info['configuration_fingerprint'])

## Frozen training configuration and fingerprint

All variants share: seed 42; maximum 100 epochs; batch size 32; AdamW LR 0.001 and weight decay 0.0001; cosine scheduler T_max 100, eta_min 0; validation Macro-F1 early stopping (max, patience 15, min_delta 0); AMP float16, initial GradScaler 1024; two DataLoader workers, persistent_workers false. No class weighting, accumulation or clipping.

The canonical fingerprint covers optimizer, scheduling, stopping, AMP/numerical policy, seed, preprocessing/augmentation, class mapping, splits and DataLoader policy. Per-run identity additionally fixes variant flags, predefined loss mode, fold, source hashes and runtime. Drift aborts execution.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(pd.Series(info['config']['training'], name='Frozen settings'))
display(info['memory'][['Variant','BatchSize','PeakAllocatedMiB','Status']])
print('Phase 11 memory preflight reused; zero optimizer steps performed in Phase 12 preparation.')

## Data pipeline and deterministic augmentation

Only Train/Validation adapters are supported. Frozen MRI preprocessing is grayscale 256×256, per-image Z-score without clipping; masks use nearest-neighbor resizing. All augmentation settings remain unchanged. The dataset receives `set_epoch(epoch - 1)` before each one-based training epoch, preserving the original zero-based Phase 6 augmentation schedule. Validation is deterministic.

Previously approved mask use inside training augmentation is retained to reject tumor-clipping transforms. A0/A1 discard auxiliary arrays before exposing the batch to the model or loss. A2–A4 retain the unchanged region, internal one-pixel boundary and normalized internal distance targets. Model forward calls receive MRI only.

## Loss functions and model factory

A0/A1 use unweighted CrossEntropyLoss only. A2–A4 reuse the exact frozen Phase 6 objective: CE + 0.3 × [Region(BCE+Dice) + Boundary(BCE+Dice) + 0.5 × SmoothL1(Distance)]. Validation preserves the same auxiliary-loss reporting as the main implementation, but model selection uses only validation classification Macro-F1.

The frozen Phase 11 factory creates each variant. No GT targets are passed to its forward interface. A5 cannot be selected by the launcher.

## Training, validation and metric utilities

`src/ablation/training.py` implements full-split epochs only; there is no shortened experimental-epoch option. Metrics use all slices exactly once per epoch. Logs record LR, total/CE loss, accuracy, macro precision/recall/F1, weighted F1, epoch duration and AMP scale. Morphology-enabled runs additionally log region, boundary, distance and weighted morphology contribution for both Train and Validation.

Nonfinite inputs, logits, loss, missing/nonfinite gradients or model parameters stop the run. The frozen main-model strict gradient policy is preserved: no clipping, accumulation or automatic hyperparameter changes. A failure record is saved; incomplete epoch state is not presented as committed history.

## Early stopping and checkpoint management

Strictly higher validation Macro-F1 updates the best checkpoint; ties retain the earlier best. Save `best_macro_f1.pt` and `last_checkpoint.pt` under `results/ablation/training/<variant>/fold_<n>/`. Each checkpoint contains variant/fold, epoch/best state, model, optimizer, scheduler, scaler, early stopper/counter, full configurations, class mapping, split/config fingerprints, RNG and loader-generator states, losses, history and timing.

Commit last atomically first, including a self-contained best snapshot and best validation predictions. An interrupted secondary best/history write can be repaired from this committed state.

## Exact resume and overwrite protection

With `RESUME=True`, restore the entire committed epoch state before continuing at epoch+1. No scheduler, optimizer, scaler, patience or random stream reset is permitted. Replaying an interrupted partial epoch starts from the previous committed epoch. The preparation check uses artificial non-experimental state in BytesIO, populated without optimizer steps; no diagnostic checkpoint is written to disk. It verifies exact restoration and next RNG draws, but does not claim an empirically tested multi-epoch replay.

`OVERWRITE=False` preserves and skips compatible completed runs. Incomplete runs require explicit resume. `RESUME` and `OVERWRITE` cannot both be true. Explicit overwrite archives the prior run within the project instead of deleting it. Active/stale run locks require review. Only selected variants/folds run, in A0→A4 and fold 1→5 order.

## Logging, summaries and plots

Real user-started experiments write `history.csv`, `run_metadata.json`, per-epoch metrics, best validation predictions, checkpoints and completion records. Completed runs get PNG/PDF loss, Macro-F1 and learning-rate curves, plus morphology curves where applicable. No curves or fake summary rows are created during preparation.

`ablation_training_summary.csv` contains only completed A0–A4 runs: variant, fold, best epoch, best validation accuracy/F1, total epochs, early stopping, compute seconds, best checkpoint path and active wall seconds. A5 is never inserted as a new run. Compute time sums completed epoch durations; active wall time excludes downtime between sessions.

## Non-training pipeline validation

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(pd.Series(preparation_audit,name='Preparation audit'))
print('No Test performance has been read. No experimental epoch or optimizer step has been executed.')

## MANUAL TRAINING — USER EXECUTION ONLY

**Cell `MANUAL-01` below is the only training-launch cell. The preparation runner never executes it.**

When ready, change only `CONFIRM_FULL_TRAINING` from `False` to `True` in this cell and run it yourself. It first validates the selected identities, test lock, fingerprints, protected artifacts, output policy and any resume state. The default selection runs only A0 Fold 1, then returns control without starting Fold 2.

For later runs, explicitly edit the variant/fold lists. To resume an incomplete run, set `RESUME=True`. Keep `OVERWRITE=False` unless deliberately archiving an old experiment. Confirmation must remain a conscious manual choice; do not use Run All after enabling it unless you intend to launch training.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
# MANUAL-01: USER EXECUTION ONLY — never executed by the preparation runner.
VARIANTS_TO_TRAIN = ["A4"]
FOLDS_TO_TRAIN = [1, 2, 3, 4, 5]

CONFIRM_FULL_TRAINING = False
RESUME = False
OVERWRITE = False

# Reload the implementation to avoid stale notebook function references.
import importlib
from pathlib import Path
from src.ablation import training as ablation_training
ablation_training = importlib.reload(ablation_training)
ROOT = Path(__import__("os").environ.get("MORPHONET_ROOT", str(Path.cwd()))).resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
completed_runs = ablation_training.train_selected(
    ROOT, variants=VARIANTS_TO_TRAIN, folds=FOLDS_TO_TRAIN,
    confirm_full_training=CONFIRM_FULL_TRAINING,
    resume=RESUME, overwrite=OVERWRITE,
)
if completed_runs:
    display(ablation_training.completion_summary(ROOT, ablation_training.preflight(ROOT)))


In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")


## Post-run verification — read only

After your selected experiment completes, this cell checks required files, finite logged values, the analytical cosine LR schedule, best validation selection and checkpoint hashes. Before any real run, it reports that no completed results exist and writes nothing.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
summary = ablation_training.completion_summary(ROOT, info, write=False)
if summary.empty:
    print('No completed ablation experiments. Training remains under user control.')
else:
    display(summary)
print('Protected artifacts verified:', check_protection(ROOT))

## Scientific audit and handoff

Preparation is complete when all safe checks pass. The delivered manual cell remains unexecuted with confirmation false. Zero experimental training runs, epochs and optimizer steps have been executed. A5, baseline artifacts and splits remain frozen. The pipeline is prepared for the user to start A0 Fold 1 manually; it has not been validated by running a real training epoch.

Sequential ablation effects remain order-dependent, with differences in capacity and computation documented in Phase 11. No component-effectiveness claim can be drawn from preparation or a single fold. Do not access Test or implement Phase 13 early. No morphology sub-ablation or lambda tuning is included.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
